# Laboratorium 10 — mały Transformer do klasyfikacji sekwencji zależnych od kolejności

Materiał towarzyszący [`README.md`](README.md). Inny, trudniejszy problem niż
wykład: sekwencje o zmiennej długości (prawdziwy padding) z losowymi dystraktorami.


In [ ]:
import time

import torch
import torch.nn as nn

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Uzywane urzadzenie: {device}")


## 1. Dane: generator zadania (gotowe + krótkie zadanie)

Każda sekwencja (długość losowa z $[10,40]$) zawiera dokładnie dwa wyróżnione
tokeny `A`, `B` w losowych pozycjach, resztę wypełniają losowe dystraktory.
Etykieta = czy `A` występuje przed `B`.


In [ ]:
N_DISTRACTORS = 10
TOKEN_A = N_DISTRACTORS
TOKEN_B = N_DISTRACTORS + 1
PAD_ID = N_DISTRACTORS + 2
VOCAB_SIZE = N_DISTRACTORS + 3
MIN_LEN, MAX_LEN = 10, 40


def make_dataset(n, seed):
    g = torch.Generator().manual_seed(seed)
    lengths = torch.randint(MIN_LEN, MAX_LEN + 1, (n,), generator=g)
    seqs = torch.full((n, MAX_LEN), PAD_ID, dtype=torch.long)
    labels = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        length = lengths[i].item()
        distractors = torch.randint(0, N_DISTRACTORS, (length,), generator=g)
        pos_ab = torch.randperm(length, generator=g)[:2]
        pa, pb = pos_ab[0].item(), pos_ab[1].item()
        distractors[pa] = TOKEN_A
        distractors[pb] = TOKEN_B
        seqs[i, :length] = distractors
        labels[i] = 1 if pa < pb else 0
    return seqs, lengths, labels


X_train, len_train, y_train = make_dataset(1500, seed=0)
X_val, len_val, y_val = make_dataset(300, seed=1)
X_test, len_test, y_test = make_dataset(300, seed=2)
print(f"train: {X_train.shape[0]} przykladow, X.shape={tuple(X_train.shape)}")
print(f"przykladowa sekwencja (dlugosc {len_train[0].item()}):")
print(X_train[0, :len_train[0]].tolist())
print(f"etykieta: {y_train[0].item()}")


**Zadanie:** zweryfikuj balans klas i niezależność długości sekwencji od etykiety.


In [ ]:
# TODO:
# 1. wypisz balans klas (y_train.float().mean())
# 2. policz srednia dlugosc sekwencji OSOBNO dla kazdej klasy (len_train[y_train==0]
#    vs len_train[y_train==1]) - powinny byc zblizone
raise NotImplementedError("TODO: weryfikacja balansu i niezaleznosci dlugosci")


## 2. Baseline bez kolejności — Twoje zadanie


In [ ]:
D_MODEL = 16


class MeanPoolBaseline(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.fc = nn.Linear(d_model, n_classes)

    def forward(self, x, lengths):
        # TODO:
        # 1. e = self.embed(x)  (B, T, d_model)
        # 2. zbuduj maske (x != PAD_ID), zeby WYKLUCZYC padding z usredniania
        # 3. policz srednia embeddingow TYLKO po prawdziwych tokenach, dzielac przez
        #    `lengths` (nie przez MAX_LEN - inaczej padding "rozwadnia" wynik
        #    krotszych sekwencji)
        # 4. zwroc self.fc(pooled)
        raise NotImplementedError("TODO: MeanPoolBaseline.forward")


## 3. Positional encoding (gotowe)

Funkcja z wykładu — nie reimplementujemy jej, tylko decydujemy, jak i gdzie ją
dołączyć (sekcja 6).


In [ ]:
def sinusoidal_positional_encoding(seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1).float()
    i = torch.arange(d_model).unsqueeze(0).float()
    angle_rates = 1.0 / torch.pow(10000, (2 * (i // 2)) / d_model)
    angles = pos * angle_rates
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(angles[:, 0::2])
    pe[:, 1::2] = torch.cos(angles[:, 1::2])
    return pe


## 4. Maska paddingu — Twoje zadanie

**Krytyczne** (zobacz `lecture.md`, "Ważna uwaga implementacyjna"): dla
`nn.MultiheadAttention`, `key_padding_mask` musi mieć konwencję `True` = to jest
padding, zignoruj — **nie** `True` = dozwolone (ta konwencja jest **odwrotna**
względem `F.scaled_dot_product_attention`).


In [ ]:
def make_key_padding_mask(x):
    # TODO: zwroc tensor bool (B, T) - True TAM GDZIE x == PAD_ID (padding,
    # do zignorowania), zgodnie z konwencja nn.MultiheadAttention
    raise NotImplementedError("TODO: make_key_padding_mask")


demo_mask = make_key_padding_mask(X_train[0:1])  # noqa: F821 - zdefiniowane wyzej
print(f"maska dla sekwencji dlugosci {len_train[0].item()} (z {MAX_LEN} pozycji):")
print(demo_mask)
print(f"liczba True (padding) w masce: {demo_mask.sum().item()} "
      f"(powinno byc {MAX_LEN - len_train[0].item()})")
assert demo_mask.sum().item() == MAX_LEN - len_train[0].item()


## 5. Blok Transformer — Twoje zadanie

Self-attention (`nn.MultiheadAttention`) + połączenie rezydualne + LayerNorm +
feed-forward + drugie połączenie rezydualne + drugi LayerNorm (Post-LN). Pamiętaj
przekazać `key_padding_mask` z sekcji 4.


In [ ]:
class TinyTransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=0.0)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, key_padding_mask):
        # TODO:
        # 1. attn_out, attn_weights = self.mha(x, x, x,
        #    key_padding_mask=key_padding_mask, need_weights=True,
        #    average_attn_weights=False)
        # 2. x = self.ln1(x + attn_out)  (polaczenie rezydualne + LayerNorm)
        # 3. x = self.ln2(x + self.ff(x))  (drugie polaczenie rezydualne + LayerNorm)
        # 4. zwroc (x, attn_weights)
        raise NotImplementedError("TODO: TinyTransformerBlock.forward")


## 6. Agregacja i klasyfikator — Twoje zadanie

`Classifier`: embedding (+ positional encoding, jeśli `use_pos_enc=True`) →
`TinyTransformerBlock` → agregacja **wykluczająca padding** → warstwa
klasyfikacyjna.


In [ ]:
N_HEADS, D_FF = 2, 32


class Classifier(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF,
                 use_pos_enc=True, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.use_pos_enc = use_pos_enc
        if use_pos_enc:
            self.register_buffer("pos_enc", sinusoidal_positional_encoding(MAX_LEN, d_model))
        self.block = TinyTransformerBlock(d_model, n_heads, d_ff)
        self.fc = nn.Linear(d_model, n_classes)

    def forward(self, x, lengths):
        # TODO:
        # 1. key_padding_mask = make_key_padding_mask(x)
        # 2. e = self.embed(x); jesli self.use_pos_enc, dodaj self.pos_enc
        # 3. out, attn_weights = self.block(e, key_padding_mask)
        # 4. zbuduj maske "prawdziwych" pozycji (~key_padding_mask) i policz srednia
        #    `out` TYLKO po tych pozycjach, dzielac przez `lengths` (NIE przez
        #    MAX_LEN - ta sama zasada co w sekcji 2)
        # 5. zwroc (self.fc(pooled), attn_weights)
        raise NotImplementedError("TODO: Classifier.forward")


**Weryfikacja (gotowa asercja): ta sama sekwencja musi dać identyczną reprezentację
niezależnie od tego, z czym jest zbatchowana.**


In [ ]:
torch.manual_seed(0)
check_model = Classifier(VOCAB_SIZE)
check_model.eval()

short_idx = (len_train < MAX_LEN).nonzero()[0, 0].item()
short_seq, short_len = X_train[short_idx:short_idx + 1], len_train[short_idx:short_idx + 1]
longer_idx = len_train.argmax().item()
longer_seq, longer_len = X_train[longer_idx:longer_idx + 1], len_train[longer_idx:longer_idx + 1]

batched_X = torch.cat([short_seq, longer_seq], dim=0)
batched_len = torch.cat([short_len, longer_len], dim=0)

with torch.no_grad():
    logits_alone, _ = check_model(short_seq, short_len)
    logits_batched, _ = check_model(batched_X, batched_len)

match = torch.allclose(logits_alone, logits_batched[0:1], atol=1e-5)
print(f"ta sama sekwencja, ten sam wynik niezaleznie od paddingu w batchu: {match}")
assert match, "BLAD: reprezentacja zalezy od paddingu - sprawdz maske (sekcja 4) i agregacje (6)"


## 7. Trening na CPU (gotowe)


In [ ]:
def train_eval(model, epochs=15, lr=0.003, batch_size=32, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n = X_train.shape[0]
    start = time.time()
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            logits, _ = model(X_train[idx], len_train[idx])
            loss = loss_fn(logits, y_train[idx])
            loss.backward()
            opt.step()
    train_time = time.time() - start
    model.eval()
    with torch.no_grad():
        val_logits, _ = model(X_val, len_val)
        val_acc = (val_logits.argmax(1) == y_val).float().mean().item()
        test_logits, _ = model(X_test, len_test)
        test_acc = (test_logits.argmax(1) == y_test).float().mean().item()
    return val_acc, test_acc, train_time


In [ ]:
baseline_model = MeanPoolBaseline(VOCAB_SIZE)  # noqa: F821 - zdefiniowane w sekcji 2
torch.manual_seed(0)
opt_b = torch.optim.Adam(baseline_model.parameters(), lr=0.003)
loss_fn_b = nn.CrossEntropyLoss()
for _ in range(15):
    perm_b = torch.randperm(X_train.shape[0])
    for i in range(0, X_train.shape[0], 32):
        idx = perm_b[i:i + 32]
        opt_b.zero_grad()
        loss_b = loss_fn_b(baseline_model(X_train[idx], len_train[idx]), y_train[idx])
        loss_b.backward()
        opt_b.step()
baseline_model.eval()
with torch.no_grad():
    baseline_val_acc = (baseline_model(X_val, len_val).argmax(1) == y_val).float().mean().item()
print(f"baseline (mean-pool, bez kolejnosci): val_acc={baseline_val_acc:.3f}")


## 8. Porównanie: z vs bez positional encoding — Twoje zadanie


In [ ]:
# TODO:
# 1. wytrenuj Classifier z use_pos_enc=True i z use_pos_enc=False (train_eval)
# 2. wypisz val_acc i test_acc dla obu
raise NotImplementedError("TODO: porownanie z/bez PE")


**Zadanie:** sprawdź na kilku ziarnach (nie jednym przebiegu), czy wynik bez PE jest
systematycznie bliski poziomowi przypadku.


In [ ]:
# TODO: powtorz porownanie z sekcji 8 dla seed=0, 1, 2 i wypisz wyniki dla kazdego
raise NotImplementedError("TODO: multi-seed check")


## 9. Zachowanie dla różnych długości sekwencji — Twoje zadanie


In [ ]:
torch.manual_seed(0)
model_with_pe = Classifier(VOCAB_SIZE, use_pos_enc=True)
train_eval(model_with_pe)
model_with_pe.eval()

# TODO:
# 1. policz predykcje modelu na X_test
# 2. dla co najmniej 3 przedzialow dlugosci (np. [10,19],[20,29],[30,40]) policz
#    accuracy TYLKO na przykladach z len_test w danym przedziale
raise NotImplementedError("TODO: accuracy wg dlugosci")


## 10. Analiza wybranych wag attention — Twoje zadanie


In [ ]:
# TODO:
# 1. wybierz jeden poprawnie sklasyfikowany przyklad testowy
# 2. znajdz pozycje tokenow A i B w tej sekwencji
# 3. wypisz wagi attention (dowolna glowa) dla zapytania = pozycja A, TYLKO dla
#    prawdziwych (nie-padding) pozycji
raise NotImplementedError("TODO: analiza wag attention")


**Zadanie (Markdown):** czy wysoka waga w stronę konkretnej pozycji jest tu "dowodem",
że model "rozumie" relację porządkową? Odwołaj się do odpowiedniego "❌/✅" z
`lecture.md`.


_(Twoja odpowiedź tutaj.)_

## 11. Ewaluacja na zbiorze testowym — Twoje zadanie


In [ ]:
# TODO:
# 1. policz test_acc dla model_with_pe (pierwszy raz uzywany tu zbior testowy do
#    OSTATECZNEJ oceny)
# 2. znajdz indeksy blednie sklasyfikowanych przykladow i ich dlugosci - czy
#    koncentruja sie przy konkretnych dlugosciach?
raise NotImplementedError("TODO: ewaluacja testowa")


## 12. Zadanie diagnostyczne

Poniższy kod **wygląda** poprawnie — definiuje maskę, buduje blok, trenuje się bez
wyjątku — ale ma błąd związany z konwencją maski.


In [ ]:
class BuggyClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.register_buffer("pos_enc", sinusoidal_positional_encoding(MAX_LEN, d_model))
        self.block = TinyTransformerBlock(d_model, n_heads, d_ff)
        self.fc = nn.Linear(d_model, 2)

    def forward(self, x, lengths):
        key_padding_mask = x != PAD_ID  # BUG: True = "to jest prawdziwy token"
        e = self.embed(x) + self.pos_enc.unsqueeze(0)
        out, _ = self.block(e, key_padding_mask)
        real_mask = (x != PAD_ID).unsqueeze(-1).float()
        pooled = (out * real_mask).sum(dim=1) / lengths.unsqueeze(1).float()
        return self.fc(pooled)


torch.manual_seed(0)
buggy_model = BuggyClassifier(VOCAB_SIZE)
opt_bug = torch.optim.Adam(buggy_model.parameters(), lr=0.003)
loss_fn_bug = nn.CrossEntropyLoss()

print("trening BuggyClassifier - obserwuj strate po kazdej epoce:")
for epoch in range(5):
    perm_bug = torch.randperm(X_train.shape[0])
    total_loss, n_batches = 0.0, 0
    for i in range(0, X_train.shape[0], 32):
        idx = perm_bug[i:i + 32]
        opt_bug.zero_grad()
        logits = buggy_model(X_train[idx], len_train[idx])
        loss = loss_fn_bug(logits, y_train[idx])
        loss.backward()
        opt_bug.step()
        total_loss += loss.item()
        n_batches += 1
    print(f"  epoka {epoch}: strata = {total_loss / n_batches}")


**Zadanie:** zdiagnozuj błąd, wyjaśnij go matematycznie (odwołując się do
`lecture.md`, sekcja "Wiersz z wszystkimi kluczami zamaskowanymi"), popraw kod,
zweryfikuj.


In [ ]:
# TODO:
# 1. jako komentarz: zdiagnozuj dokladnie, ktora linia jest zrodlem bledu i dlaczego
# 2. sprawdz: czy w train sa sekwencje dlugosci MAX_LEN (len_train == MAX_LEN)?
# 3. napisz FixedClassifier - poprawiona wersja BuggyClassifier
# 4. wytrenuj FixedClassifier, przy KAZDYM kroku sprawdz torch.isfinite(loss)
# 5. porownaj koncowa val_acc z oryginalnym, poprawnym Classifier
raise NotImplementedError("TODO: diagnoza i poprawka")


## 13. Wnioski — Twoje zadanie


**1. Które elementy architektury (sekcja 5-6) odpowiadają za: (a) mieszanie informacji między pozycjami, (b) wprowadzenie informacji o pozycji, (c) redukcję sekwencji do jednego wektora?**

_(Twoja odpowiedź tutaj.)_

**2. Jak duża jest różnica accuracy między baseline'em (sekcja 2) a Transformerem z PE (sekcja 8)?**

_(Twoja odpowiedź tutaj.)_

**3. Czy wynik "bez PE = poziom przypadku" jest stabilny na kilku ziarnach?**

_(Twoja odpowiedź tutaj.)_

**4. Czy maskowanie jest poprawne — czy masz dowód, że model nie korzysta z informacji zawartej w paddingu?**

_(Twoja odpowiedź tutaj.)_

**5. Co dokładnie pokazuje (a czego nie pokazuje) ten eksperyment o mechanizmie attention?**

_(Twoja odpowiedź tutaj.)_

**6. Jakie jest co najmniej jedno ograniczenie tego eksperymentu?**

_(Twoja odpowiedź tutaj.)_